<a href="https://colab.research.google.com/github/RobJavVar/DataSciencePsychNeuro/blob/master/ExerciseSubmissions/12_cross-validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 12: Cross validation
-----

In this exercise, we'll practice implementing cross validation techniques, including leave-one-out and k-fold cross validation. We'll use the `PimaIndiansDiabetes2` practice dataset, which has medical data on a group of Pima Native American women, including whether or not they have diabetes. This dataset is part of the `mlbench` package. We'll be using each person's medical history to predict whether or not they have been diagnosed with diabetes.

# 1: Data (1 pts)
---

Load the `tidyverse`, `boot`, and `mlbench` packages (you may need to install `boot` and `mlbench`).

Load the `PimaIndiansDiabetes2` dataset using the `data()` function. Drop the `insulin` column (it just has a lot of missing data) and then drop `NA`s from the rest of the dataset. Save your updated dataset to a new variable name. Finally, print the dimensions of your new dataset, and look at the first few lines of data.

In [6]:
library(tidyverse)
library(boot)
library(mlbench)

── Attaching core tidyverse packages ──────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [23]:
data("PimaIndiansDiabetes2")

In [24]:
pima <- PimaIndiansDiabetes2
head(pima)

,pregnant,glucose,pressure,triceps,insulin,mass,pedigree,age,diabetes
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
1,6,148,72,35,NA,33.6,0.627,50,pos
2,1,85,66,29,NA,26.6,0.351,31,neg
3,8,183,64,NA,NA,23.3,0.672,32,pos
4,1,89,66,23,94,28.1,0.167,21,neg
5,0,137,40,35,168,43.1,2.288,33,pos
6,5,116,74,NA,NA,25.6,0.201,30,neg


In [25]:
pima <- pima %>%
    drop_na()%>%
    select(-"insulin")

In [26]:
dim(pima)
head(pima)

[1] 392   8

,pregnant,glucose,pressure,triceps,mass,pedigree,age,diabetes
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
4,1,89,66,23,28.1,0.167,21,neg
5,0,137,40,35,43.1,2.288,33,pos
7,3,78,50,32,31.0,0.248,26,pos
9,2,197,70,45,30.5,0.158,53,pos
14,1,189,60,23,30.1,0.398,59,pos
15,5,166,72,19,25.8,0.587,51,pos


(Note that in medical contexts, `pedigree` refers to a system of measuring family history of a condition. So here, higher numbers mean greater family history of diabetes. You can read more about this dataset [here](https://rdrr.io/cran/mlbench/man/PimaIndiansDiabetes.html).)

# 2. Leave-one-out Cross Validation (4 pts)

In the tutorial, we learned how to fit leave-one-out cross validation using the `cv.glm` function from the `boot` package. But we can also do this manually using `predict()` like we have in the past.

Let's predict `diabetes`, a dichotomous outcome, using all the other variables in our modified dataset.

First, fit a logistic regression model using all of the observations except the very first one. Then use your fitted model to predict whether your holdout case is positive or negative for diabetes. Remember that logistic regression coefficients are in **log-odds**, meaning that if an output is positive, the probability of the outcome is greater than 50%; if the output is negative, the probability of the outcome is less than 50%.

Compare your result to the actual response in row one above. Did your model correctly classify this observation?

In [33]:
# logistic regression
glm.pima = glm(diabetes ~ glucose + pressure+triceps+mass+pedigree+age, data = pima, family = binomial)
coef(glm.pima)

(Intercept)       glucose      pressure       triceps          mass 
-9.9836927019  0.0362171892 -0.0005803995  0.0120711367  0.0631862777 
     pedigree           age 
 1.0630391908  0.0516373952

In [34]:
#Predict if pos or neg for diabetes
predict.prima = predict(glm.pima)
head(predict.prima)

4         5         7         9        14        15 
-3.483586  2.236932 -3.236515  2.485590  2.475771  1.103641

In [37]:
#Convert to probability
pred_prob <- plogis(predict.prima)
pred_class <- ifelse(pred_prob > 0.5, "pos", "neg")
head(pred_class)

4     5     7     9    14    15 
"neg" "pos" "neg" "pos" "pos" "pos"

In comparison to the inital ouptput under the diabest column, yes it is correctly classifying the observations

So we just calculated a single iteration of LOOCV. We used 531 rows of our data to fit a model to predict the outcome of the last row.

Below, use a `for` loop to iterate through the rest of your dataset doing the same thing. You will need to:
* Create a data frame `results` with two columns: one named `actual` which holds the true classification for each observation, and one named `predicted`, which should be filled with `NA`s. This is where you'll store the output of your loop.
* Create a loop that runs through each row of your data, pulls that observation out, trains your model on the remaining data, and then tests the fitted model on your test observation.
* Store your model *predictions* ("pos" or "neg" -- not the log-odds) in the `predicted` column of your `results` dataframe

After you run your loop, print the first few lines of `results`.

In [38]:
# Initialize `results` data frame
results <- data.frame(
  actual    = pima$diabetes,
  predicted = NA
)


# LOOCV loop
for (i in 1:nrow(pima)) {
  

}

#for loop
for (i in 1:nrow(pima)){ #don't forget to change this to your data set name
    # separate individual observation `i` from the rest of your data
    holdout <- pima[i, ]
  training <- pima[-i, ]

  model_loo <- glm(diabetes ~ ., data = training, family = binomial)
  
  # test model on hold out observation
  pred_logodds <- predict(model_loo, newdata = holdout)
  
  # classify model prediction as "pos" or "neg" and add to results
  results$predicted[i] <- ifelse(plogis(pred_logodds) > 0.5, "pos", "neg")
  
}


Now, calculate the overall error of your model. What proportion of cases were incorrectly classified?

In [41]:
error_rate <- mean(results$actual != results$predicted)
print(error_rate) #22% of cases were incorrectly classified. 

[1] 0.2219388


# 3. Compare to `cv.glm` (3 pts)

Now, let's compare this result to the `cv.glm` function. Using the tutorial as a guide, use `cv.glm` to run LOOCV on the data, using the same model (i.e., still using all of the variables to predict diabetes diagnosis).

Note that, because this is a `classification` problem and not a regression problem like in the tutorial, we need to adjust the `cost` argument of `cv.glm`. We can read more about this in the docs:

In [ ]:
#?cv.glm

Here, we see `cost` is defined as:
> "A function of two vector arguments specifying the cost function for the cross-validation. The first argument to cost should correspond to the **observed responses** and the second argument should correspond to the **predicted or fitted responses** from the generalized linear model."

In the example code (scroll to bottom of the docs), we see that the appropriate cost function for a binary classification is

``
cost <- function(r, pi = 0) mean(abs(r-pi) > 0.5)
``

Where `r` is the vector of observed responses (technically "pos" and "neg", but R treats these as 1 and 0 under the hood), and `pi` is the vector of *probabilities* (not log-odds) fit by the model. Thus, this boils down to our error: what proportion of observations were incorrectly classified. You will need to include this code below.

In [74]:
new.cost <- function(r, pi = 0) {
  mean(abs(r - pi) > 0.5)
}

In [75]:
fullmodel <- glm(diabetes ~ ., data = pima, family = binomial)

In [77]:
loocv <- cv.glm(pima, fullmodel, cost = new.cost)
cat("LOOCV error:", loocv$delta[1], "\n")

LOOCV error: 0.2219388 


How do your results compare to your manual LOOCV above?
>I obtained the same result as the manual loocv done

# 4. Adjusting K and Reflection (2 pts)

Recall that LOOCV has some drawbacks. In particular, it has quite high *variance* which can lead to poor performance on new test data. We can reduce this variance by increasing K.

Below, re-run your cross validation using `cv.glm` with `k` set to 3, 5, 10, and 15.

In [69]:
set.seed(1)
#INSERT CODE BELOW

# K = 3
cv_k3  <- cv.glm(pima, fullmodel, cost = new.cost, K = 3)

# K = 5
cv_k5  <- cv.glm(pima, fullmodel, cost = new.cost, K = 5)

# K = 10

cv_k10  <- cv.glm(pima, fullmodel, cost = new.cost, K = 10)

# K = 15
cv_k15 <- cv.glm(pima, fullmodel, cost = new.cost, K = 15)


In [80]:
cat("K=3 error:", cv_k3$delta[1], "\n")
cat("K=5 error:", cv_k5$delta[1], "\n")
cat("K=10 error:", cv_k10$delta[1], "\n")
cat("K=15 error:", cv_k15$delta[1], "\n")

K=3 error: 0.2193878 
K=5 error: 0.2193878 
K=10 error: 0.2193878 
K=15 error: 0.2168367 


loocv error: 0.2219388   

#### Reflection

How do your errors compare to your LOOCV error above? How do they change as k increases?
> I got identical results as the loocv error above and even for k=3,5,and 10. The error decreased slightly at k=15 reflecting a slight lower variance so, I would still interpret this to show that even among higher folds the error rate remained consistent in reflect variance. 
>
> *  

If you change the random seed above, you'll get slightly different errors. If you were to do the same with your LOOCV above , would you expect to get different results each time? Why or why not?
> I would expect to get the same result each time because loocv does not involve any random shuffling so I would expect no change in results with loocv but I would 
> *

**DUE:** 5pm March 19, 2026 

**IMPORTANT** Did you collaborate with anyone on this assignment? If so, list their names here.
> *Someone's Name*
>
>